# imports

In [3]:
import os
from glob import glob 
import subprocess as sp 
import boto3
from botocore import UNSIGNED
from botocore.client import Config
import pandas as pd 
import numpy as np 
import multiprocessing as mp 
from functools import partial 
import xarray as xr 
import json 
import warnings
warnings.filterwarnings("ignore")

# download data

In [2]:
# util funcs 
def avail_s3_keys(bucket, obj1, obj2, input_interval, output_times):
    
    tid_dst = output_times 
    dates = tid_dst.normalize().unique()  

    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
    paginator = s3.get_paginator('list_objects_v2') 
    tid_src = [] 
    
    # available times 
    for d in dates:
        date = d.strftime("%Y%m%d") 
        prefix = f'{obj1}/{obj2}/{date}/'
        for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
            for obj in page.get('Contents', []):
                key = obj['Key']
                filename = os.path.basename(key)
                if filename:   # skip directories
                    file_time = filename[-24:-9]  # 'MRMS_Obj_YYYYMMDD-HHMMSS.grib2.gz'
                    tid_src.append(pd.to_datetime(file_time, format='%Y%m%d-%H%M%S').to_numpy())

    # target times 
    tid_src = np.array(tid_src)
    idx = np.searchsorted(tid_src, tid_dst)
    idx = np.clip(idx, 1, len(tid_src) - 1)
    left  = tid_src[idx - 1]
    right = tid_src[idx]
    closest_idx = np.where(np.abs(tid_dst - left) <= np.abs(tid_dst - right), idx - 1, idx)

    timestamp = np.array([tid_src[closest_idx[i]] for i in range(len(tid_dst))])
    delta_t = timestamp - tid_dst
    timestamp_out = timestamp[abs(delta_t) < np.timedelta64(input_interval, 'm')]   

    timestamp_out_dt = pd.to_datetime(timestamp_out)
    key_str = [f'{obj1}/{obj2}/{t:%Y%m%d}/MRMS_{obj2}_{t:%Y%m%d-%H%M%S}.grib2.gz' for t in timestamp_out_dt]
    return key_str     

 
def download_s3(bucket, savedir, key):    
    local_path = os.path.join(savedir, os.path.basename(key))    
    s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))
        
    uncompressed_path = local_path.replace('.gz', '')
    file_exists = (os.path.isfile(local_path) and os.path.getsize(local_path) > 0) or \
                  (os.path.isfile(uncompressed_path) and os.path.getsize(uncompressed_path) > 0)    
    if file_exists:
        return    
    s3.download_file(bucket, key, local_path)
    
    
def parallel_download_s3(bucket, savedir, keys, num_workers=8, func=None):
    if func is None:
        func = download_s3
    worker_func = partial(func, bucket, savedir) 
    with mp.Pool(num_workers) as pool:
        pool.imap(worker_func, keys)
        pool.close()
        pool.join() 
    

In [4]:
# user inputs
bucket_name = 'noaa-mrms-pds' 
obj_area = 'CONUS'  # 'CONUS', 'ALASKA', etc. 
start_time = '20210401 00:00:00'
end_time = '20211001 00:00:00'
high_freq_out = '5min'
save_dir = '/pscratch/sd/i/iclas2/meng/mrms/CONUS_2021'
num_workers = mp.cpu_count()  

# === parallel download ===
# precipitation (hourly)
obj_product = 'MultiSensor_QPE_01H_Pass2_00.00'
obs_interval_min = 60   # source time res in minutes 
toi = pd.date_range(start_time, end_time, freq='1h')  # target output times 
local_dir = f'{save_dir}/{obj_product}'  
os.makedirs(local_dir, exist_ok=True)
keys = avail_s3_keys(bucket_name, obj_area, obj_product, obs_interval_min, toi) 
with open(f'{local_dir}/keys.json', "w") as f:
    json.dump(keys, f)
# serial download 
# download_s3(bucket_name, local_dir, keys[0]) 
parallel_download_s3(bucket_name, local_dir, keys, num_workers=num_workers) 
print(f'Downloaded {len(keys)} files for {obj_product} [ref: {len(toi)}]') 

# 3d reflectivity
heights = [
    '00.50', '00.75', '01.00', '01.25', '01.50', '01.75', '02.00', '02.25', '02.50', '02.75', 
    '03.00', '03.50', '04.00', '04.50', '05.00', '05.50', '06.00', '06.50', '07.00', '07.50',
    '08.00', '08.50', '09.00', '10.00', '11.00', '12.00', '13.00', '14.00', '15.00', '16.00',
    '17.00', '18.00', '19.00',
]
obs_interval_min  = 2  
toi = pd.date_range(start_time, end_time, freq=high_freq_out)  # target output times 
for z in heights:
    obj_product = f'MergedReflectivityQC_{z}' 
    local_dir = f'{save_dir}/{obj_product}'  
    os.makedirs(local_dir, exist_ok=True) 
    keys = avail_s3_keys(bucket_name, obj_area, obj_product, obs_interval_min, toi)
    with open(f'{local_dir}/keys.json', "w") as f:
        json.dump(keys, f)
    parallel_download_s3(bucket_name, local_dir, keys, num_workers=num_workers) 
    print(f'Downloaded {len(keys)} files for {obj_product} [ref: {len(toi)}]') 

# radar-only rain rate 
obj_product = 'PrecipRate_00.00'
obs_interval_min = 2   # source time res in minutes
local_dir = f'{save_dir}/{obj_product}'  
os.makedirs(local_dir, exist_ok=True)
keys = avail_s3_keys(bucket_name, obj_area, obj_product, obs_interval_min, toi) 
with open(f'{local_dir}/keys.json', "w") as f:
    json.dump(keys, f)
parallel_download_s3(bucket_name, local_dir, keys, num_workers=num_workers)
print(f'Downloaded {len(keys)} files for {obj_product} [ref: {len(toi)}]') 


Downloaded 4393 files for MultiSensor_QPE_01H_Pass2_00.00 [ref: 4393]


# stitch per-level reflectivity into a single file

In [ ]:
def convert_grib2_to_netcdf(time, workdir, outdir, logdir, heights):
    outpath = os.path.join(outdir, f'MRMS_MergedReflectivityQC_L33_{time}.nc')
    logpath = os.path.join(logdir, f'{time}.log') 
    if os.path.exists(outpath) and os.path.getsize(outpath) > 250 * 1024: # 25 Mb 
        print(f"{outpath} already exists, skipping.")
        return
    else:
        # Uncompress files if needed
        for z in heights:
            fgz = glob(f'{workdir}/MergedReflectivityQC_{z}/MRMS_MergedReflectivityQC_{z}_{time[:-2]}??.grib2.gz')
            if fgz and not os.path.exists(fgz[0].replace('.gz', '')):
                sp.run(f"gunzip {fgz[0]}", shell=True)
                
        # Gather input files and track missing heights 
        infiles = []
        missing_heights = []
        for z in heights:
            files = glob(f'{workdir}/MergedReflectivityQC_{z}/MRMS_MergedReflectivityQC_{z}_{time[:-2]}??.grib2')
            if files:
                infiles.append(files[0])
            else:
                missing_heights.append(z)
        
        ds = xr.open_mfdataset(infiles, concat_dim='heightAboveSea', combine='nested')     
        ds = ds.rename({'unknown': 'Reflectivity'})
        
        height_values = np.array(heights, dtype=float) * 1000 # Convert to meters
        dsout = ds.reindex(heightAboveSea=height_values, fill_value=-999.)  
        
        # Add time dimension and metadata
        dsout['Reflectivity'] = dsout['Reflectivity'].expand_dims('time')
        dsout['Reflectivity'].attrs['_NoEchoValue'] = -99.
        dsout['Reflectivity'].attrs['units'] = 'dBZ'
        dsout['Reflectivity'].attrs['long_name'] = 'MergedReflectivityQC' 
        dsout['Reflectivity'].attrs['standard_name'] = 'MergedReflectivityQC'
        if missing_heights:
            msg = ','.join(missing_heights)
            dsout['Reflectivity'].attrs['missing_levels'] = msg 
            with open(logpath, 'w') as f:
                f.write(msg + '\n') 
        attrs_to_remove = [attr for attr in dsout['Reflectivity'].attrs if attr.startswith('GRIB')]
        for attr in attrs_to_remove:
            del dsout['Reflectivity'].attrs[attr]
            
        dsout['time'] = [pd.to_datetime(time, format='%Y%m%d-%H%M%S')]
        dsout = dsout.drop_vars([v for v in dsout.coords if v not in ['time', 'longitude', 'latitude', 'heightAboveSea']])

        comp = {'_FillValue': -999., 'zlib': True, 'complevel': 1, 'shuffle': True}
        encoding = {'Reflectivity': comp}        
        dsout.to_netcdf(outpath, encoding=encoding, unlimited_dims='time') 
        print(f"Saved {outpath}") 
        

case ='CONUS_2021'
workdir = '/pscratch/sd/i/iclas2/meng/mrms'
heights = [
    '00.50', '00.75', '01.00', '01.25', '01.50', '01.75', '02.00', '02.25', '02.50', '02.75', 
    '03.00', '03.50', '04.00', '04.50', '05.00', '05.50', '06.00', '06.50', '07.00', '07.50',
    '08.00', '08.50', '09.00', '10.00', '11.00', '12.00', '13.00', '14.00', '15.00', '16.00',
    '17.00', '18.00', '19.00'
]

with open(f'{workdir}/{case}/MergedReflectivityQC_{heights[0]}/keys.json') as f:
    files = json.load(f)
times = [os.path.basename(f)[-24:-9] for f in files] 
outdir = os.path.join(f'{workdir}/{case.lower()}_netcdf', 'tmp') 
os.makedirs(outdir, exist_ok=True) 
logdir = os.path.join(f'{workdir}/{case.lower()}_netcdf', 'logs')
os.makedirs(logdir, exist_ok=True) 

# time = times[9]
time = '20210418-030437'  #20250401-092042
# convert_grib2_to_netcdf(time, f'{workdir}/{case}', outdir, logdir, heights)

In [ ]:
worker_func = partial(convert_grib2_to_netcdf, workdir=f'{workdir}/{case}', heights=heights, outdir=outdir, logdir=logdir)
num_workers = 4
with mp.Pool(num_workers) as pool:
    pool.imap(worker_func, times[:num_workers])
    pool.close()
    pool.join() 

# convert `MultiSensor_QPE_01H_Pass2` to netcdf 

In [ ]:
def convert_grib2_to_netcdf_precip(time_in, workdir, outdir, logdir):
    time = time_in.strftime("%Y%m%d-%H%M%S") 
    outpath = os.path.join(outdir, f'MRMS_MultiSensor_QPE_01H_Pass2_00.00_{time}.nc')
    logpath = os.path.join(logdir, f'{time}.log') 
    if os.path.exists(outpath):  
        print(f"{outpath} already exists, skipping.")
        return
    else:
        # Uncompress file if needed
        fname = f'{workdir}/MultiSensor_QPE_01H_Pass2_00.00/MRMS_MultiSensor_QPE_01H_Pass2_00.00_{time}.grib2'
        fgz = glob(f'{fname}.gz')
        if (not fgz) and (not os.path.exists(fname)):
            with open(logpath, 'w') as f:
                f.write('missing\n') 
            return
        else: 
            if fgz and not os.path.exists(fname):
                sp.run(f"gunzip {fgz[0]}", shell=True)
            else:
                mrms_ds = xr.open_dataset(fname)
                dsout = mrms_ds.rename({'unknown': 'precip'})
                dsout['precip'] = dsout['precip'].expand_dims('time')
                dsout['precip'].attrs['_NoValue'] = -1
                dsout['precip'].attrs['units'] = 'mm/hr'
                dsout['precip'].attrs['long_name'] = 'Multi-sensor accumulation 1-hour (1-hour latency)'
                dsout['precip'].attrs['standard_name'] = 'MultiSensor_QPE_01H_Pass2_00.00'
                
                attrs_to_remove = [attr for attr in dsout['precip'].attrs if attr.startswith('GRIB')]
                for attr in attrs_to_remove:
                    del dsout['precip'].attrs[attr]
                    
                dsout['time'] = [pd.to_datetime(time, format='%Y%m%d-%H%M%S')]
                dsout = dsout.drop_vars([v for v in dsout.coords if v not in ['time', 'longitude', 'latitude', 'heightAboveSea']])
                                
                comp = {'_FillValue': -3, 'zlib': True, 'complevel': 5, 'shuffle': True}
                encoding = {'precip': comp}
                dsout.to_netcdf(outpath, encoding=encoding, unlimited_dims='time')
                print(f"Saved {outpath}")        
        
            
case ='CONUS_2025'
bucket_name = 'MultiSensor_QPE_01H_Pass2_00.00'
start_time = '20250401 00:00:00'
end_time = '20251001 00:00:00'
times = pd.date_range(start_time, end_time, freq="1h")

workdir = '/pscratch/sd/i/iclas2/meng/mrms'
outdir = os.path.join(f'{workdir}/{case.lower()}_netcdf', bucket_name) 
os.makedirs(outdir, exist_ok=True) 
logdir = os.path.join(f'{workdir}/{case.lower()}_netcdf', 'logs_precip')
os.makedirs(logdir, exist_ok=True) 

worker_func = partial(convert_grib2_to_netcdf_precip, workdir=f'{workdir}/{case}', outdir=outdir, logdir=logdir)
num_workers = 64
with mp.Pool(num_workers) as pool:
    pool.imap(worker_func, times)
    pool.close()
    pool.join() 
    


# convert `PrecipRate_00.00` to netcdf 

In [ ]:
def convert_grib2_to_netcdf_precipRate(infile, outdir):
    if infile.endswith('.gz'):
        sp.run(f"gunzip {infile}", shell=True)
        infile = infile.replace('.gz', '')
    
    time = os.path.basename(infile)[-21:-6]
    outpath =  os.path.join(outdir, f'MRMS_PrecipRate_00.00_{time}.nc') 
    if not os.path.exists(outpath):
        ds = xr.open_dataset(infile) 
        dsout = ds.rename({'unknown': 'precip'})
        dsout['precip'] = dsout['precip'].expand_dims('time')
        dsout['precip'].attrs['_NoValue'] = -1
        dsout['precip'].attrs['units'] = 'mm/hr'
        dsout['precip'].attrs['long_name'] = 'Radar Precipitation Rate'
        dsout['precip'].attrs['standard_name'] = 'PrecipRate_00.00'

        
        attrs_to_remove = [attr for attr in dsout['precip'].attrs if attr.startswith('GRIB')]
        for attr in attrs_to_remove:
            del dsout['precip'].attrs[attr]
                
        dsout['time'] = [pd.to_datetime(time, format='%Y%m%d-%H%M%S')]
        dsout = dsout.drop_vars([v for v in dsout.coords if v not in ['time', 'longitude', 'latitude', 'heightAboveSea']])
                        
        comp = {'_FillValue': -3, 'zlib': True, 'complevel': 5, 'shuffle': True}
        encoding = {'precip': comp}
        dsout.to_netcdf(outpath, encoding=encoding, unlimited_dims='time')
        print(f"Saved {outpath}")         
        
            
case ='CONUS_2025'
bucket_name = 'PrecipRate_00.00'

workdir = '/pscratch/sd/i/iclas2/meng/mrms'
outdir = os.path.join(f'{workdir}/{case.lower()}_netcdf', bucket_name) 
os.makedirs(outdir, exist_ok=True) 

infiles = sorted(glob(f'{workdir}/{case}/{bucket_name}/MRMS_PrecipRate_00.00_*'))
infiles = [f for f in infiles if not f.endswith('.idx')]

worker_func = partial(convert_grib2_to_netcdf_precipRate, outdir=outdir)
num_workers = 32
with mp.Pool(num_workers) as pool:
    pool.imap(worker_func, infiles)
    pool.close()
    pool.join() 
    
